# Gold volatility baseline forecasts

This notebook loads the prepared gold-futures dataset, computes baseline variance forecasts at each forecast origin, and compares each model with the actual realized-variance target. Each model gets its own chart panel, followed by MAE/RMSE summary metrics.

In [1]:
from pathlib import Path

from gold_vol_pipeline import prediction_history

repo_root = Path.cwd()
if not (repo_root / "data" / "processed" / "gold_futures.csv").exists():
    repo_root = repo_root.parent
data_path = repo_root / "data" / "processed" / "gold_futures.csv"
plot_dir = repo_root / "plots" / "model_comparisons"
print(f"Using data file: {data_path}")


Using data file: /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/data/processed/gold_futures.csv


In [2]:
import pandas as pd

df = pd.read_csv(data_path)
df.head()


,date,close,log_return,target_start_date,target_end_date,realized_variance,source,instrument,currency,price_unit
0,2015-01-05,1203.900024,0.014980,2015-01-06,2015-01-12,0.000442,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,USD per troy ounce
1,2015-01-06,1219.300049,0.012711,2015-01-07,2015-01-13,0.000282,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,USD per troy ounce
2,2015-01-07,1210.599976,-0.007161,2015-01-08,2015-01-14,0.000230,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,USD per troy ounce
3,2015-01-08,1208.400024,-0.001819,2015-01-09,2015-01-15,0.000815,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,USD per troy ounce
4,2015-01-09,1216.000000,0.006270,2015-01-12,2015-01-16,0.000868,Yahoo Finance,"COMEX Gold Futures (continuous series, GC=F)",USD,USD per troy ounce


In [3]:
comparison = prediction_history(
    str(data_path),
    horizon=5,
    sample_size=500,
)
comparison.head()


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/optimize/_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


,date,actual,historical_variance,rolling_variance,ewma_variance,ar1_squared_return,ma1_squared_return,arch1,garch11
0,2024-11-07,0.000964,0.000432,0.000485,0.000502,0.000436,0.000429,0.000431,0.000566
1,2024-11-11,0.000401,0.000433,0.000647,0.000701,0.000524,0.000504,0.000490,0.000558
2,2024-11-12,0.000421,0.000433,0.000646,0.000664,0.000425,0.000421,0.000428,0.000622
3,2024-11-13,0.000424,0.000433,0.000655,0.000642,0.000429,0.000432,0.000421,0.000556
4,2024-11-18,0.001604,0.000433,0.000707,0.000629,0.000458,0.000453,0.000442,0.000571


In [4]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plot_dir.mkdir(parents=True, exist_ok=True)
model_columns = [
    'historical_variance',
    'rolling_variance',
    'ewma_variance',
    'ar1_squared_return',
    'ma1_squared_return',
    'arch1',
    'garch11',
]
metrics = pd.DataFrame([
    {
        'model': model,
        'MAE': (comparison[model] - comparison['actual']).abs().mean(),
        'RMSE': ((comparison[model] - comparison['actual']) ** 2).mean() ** 0.5,
    }
    for model in model_columns
]).sort_values('RMSE')
display(metrics)

plot_paths = []
for model in model_columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(comparison['date'], comparison['actual'], label='actual', color='black', linewidth=1.5)
    ax.plot(comparison['date'], comparison[model], label=f'{model} forecast', alpha=0.85)
    ax.set_title(f'Actual vs {model} forecast (5-session realized variance)')
    ax.set_xlabel('Forecast origin date')
    ax.set_ylabel('Realized variance')
    ax.legend(loc='upper right')
    plt.xticks(rotation=45)
    fig.tight_layout()
    output_path = plot_dir / f'{model}_vs_actual.png'
    fig.savefig(output_path, dpi=200)
    plt.close(fig)
    plot_paths.append(output_path)
    display(Image(filename=str(output_path)))

print('Saved comparison charts:')
for output_path in plot_paths:
    print(output_path)


,model,MAE,RMSE
6,garch11,0.000783,0.001891
2,ewma_variance,0.000921,0.001911
3,ar1_squared_return,0.000864,0.002003
5,arch1,0.000868,0.002008
4,ma1_squared_return,0.000868,0.002009
1,rolling_variance,0.000977,0.002013
0,historical_variance,0.000874,0.002018


/Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/plots/prediction_vs_actual.png


/var/folders/fq/_cj24jgn4rzdmnbs9wqpq60m0000gn/T/ipykernel_66636/953627431.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
